## Separação Meses
- Obter o arquivo bruto CSV do PSTools
- Separar dados pelo ultimo mês (já foi realizada o fatimaneto histórico)
- Normalizar Ids Incorretos, manter somente os ids com 9 digitos e os demais são do OOH

In [ ]:
import pandas as pd

In [ ]:
%pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.4 MB/s eta 0:00:00


Exception ignored in: <function ZipFile.__del__ at 0x79c11f573ce0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: I/O operation on closed file


In [2]:
def carregamento(prefixo, path):
    import pandas as pd
    import glob
    import os

    # Busca CSV e Excel
    arquivos = glob.glob(os.path.join(path, f"{prefixo}*.csv")) + \
               glob.glob(os.path.join(path, f"{prefixo}*.xlsx")) + \
               glob.glob(os.path.join(path, f"{prefixo}*.xls"))

    dfs = []

    for arq in arquivos:
        extensao = os.path.splitext(arq)[1].lower()

        if extensao == ".csv":
            df = pd.read_csv(arq, sep=';', low_memory=False)

        elif extensao in [".xlsx", ".xls"]:
            df = pd.read_excel(arq)

        else:
            continue  # ignora formatos desconhecidos

        dfs.append(df)

    if not dfs:
        raise ValueError("Nenhum arquivo encontrado para o prefixo informado.")

    df_final = pd.concat(dfs, ignore_index=True)

    return df_final

In [3]:
import functions
rep7 = functions.carregamento(
        path='C:/Users/70089581/Kantar/BKO - dados-ops/do_rep7/bruto',
        prefixo='Brasil-Reporte-7'
    )

rep7 = functions.normalize(rep7)

Dimensão entrada saida iguais



In [9]:
rep7.loc[rep7['Unnamed: 9'].isna()].value_counts()

Series([], Name: count, dtype: int64)

In [ ]:
csv_bs = carregamento('Brasil-Reporte-7')
csv_bs.DiaDeCompra = pd.to_datetime(csv_bs.DiaDeCompra, format='%d/%m/%Y')
csv_bs['mes_ref'] = csv_bs.DiaDeCompra.dt.to_period('M')
csv_bs['ano_ref'] = csv_bs.DiaDeCompra.dt.to_period('Y')
csv_bs['len_us'] = csv_bs['UserPS'].str.len()

### Fatiamento

In [ ]:
tt=csv_bs.shape[0]
tt2=csv_bs.duplicated().sum()

f"{tt2/tt:.2f}%"

'0.00%'

In [ ]:
bs_9 = csv_bs.loc[csv_bs.len_us == 9].copy()
bs_8 = csv_bs.loc[csv_bs.len_us == 8].copy()
bs_14 = csv_bs.loc[csv_bs.len_us == 14].copy() # Mesclado OOH & IHS + IBS
bs_7 = csv_bs.loc[csv_bs.len_us == 7].copy()
bs_6 = csv_bs.loc[csv_bs.len_us == 6].copy()

bs_10 = csv_bs.loc[csv_bs.len_us == 10].copy() # prefixo br-OOH
bs_11 = csv_bs.loc[csv_bs.len_us == 11].copy() # prefixo br-OOH
bs_12 = csv_bs.loc[csv_bs.len_us == 12].copy() # prefixo br-OOH
bs_13 = csv_bs.loc[csv_bs.len_us == 13].copy() # prefixo br-OOH
bs_15 = csv_bs.loc[csv_bs.len_us == 15].copy() # prefixo br-OOH

bs_8.loc[(bs_8.len_us == 8), 'UserPS'] = (bs_8.UserPS.astype(int) - 55000000) + 550000000

mask = ~(bs_14['Prefijo'].isin(['OOH']))

bs_14.loc[mask, 'UserPS'] = (
    bs_14.loc[mask, 'UserPS']
    .str.replace('ih', '', regex=False)
    .str.split('-', n=1).str[0]
)

bs_7.loc[(bs_7.len_us == 7), 'UserPS'] = (bs_7.UserPS.astype(int) - 5500000) + 550000000

bs_6.loc[(bs_6.len_us == 6), 'UserPS'] = (bs_6.UserPS.astype(int) - 550000) + 550000000

base = pd.concat([bs_9, bs_8, bs_14, bs_7, bs_6, bs_10, bs_11, bs_12, bs_13, bs_15])

base['len_us'] = base['UserPS'].str.len()


base.len_us.value_counts()


,count
len_us,
9.0,60992
11.0,49625
13.0,11478
14.0,9797
15.0,4196
12.0,3264
10.0,2680


In [ ]:
csv_bs.shape

(152616, 13)

In [ ]:
base.shape

(152616, 13)

In [ ]:
import pandas as pd
import zipfile
from io import BytesIO

# 1. Pegar apenas o mês mais recente
mes_recente = base['mes_ref'].max()
base_mes = base.loc[base['mes_ref'] == mes_recente]

# 2. Nome do ZIP
zip_filename = 'do_rep7_pstools_br.zip'

with zipfile.ZipFile(zip_filename, mode='w', compression=zipfile.ZIP_DEFLATED) as zf:

    # Nome do arquivo Excel
    nome_arquivo = f'do_rep7_pstools_br_{mes_recente}.xlsx'

    # 3. Criar Excel em memória
    excel_buffer = BytesIO()

    with pd.ExcelWriter(excel_buffer, engine='xlsxwriter') as writer:
        base_mes.to_excel(writer, index=False, sheet_name='dados')

    # 4. Escrever no ZIP
    zf.writestr(nome_arquivo, excel_buffer.getvalue())

print(f'ZIP criado com sucesso: {zip_filename}')

ZIP criado com sucesso: do_rep7_pstools_br.zip


# Identificação IDs Top Client
- Obter Ids pré-alocados;
- Carregamento dos dados já tratados

In [ ]:
def extrair_zip(zip_path, extract_path):
    import zipfile
    import os
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("Arquivos extraídos em:", extract_path)

    return os.listdir(extract_path)

extrair_zip(zip_path = 'do_rep7_pstools_br.zip', extract_path ='/content/raw')

Arquivos extraídos em: /content/raw


['do_rep7_pstools_br_2026-04.xlsx']

In [4]:
import pandas as pd

df_rep7 = carregamento('do_rep7', path = '/content/raw/')
df_rep7['len_us'] = df_rep7['UserPS'].str.len()
df_rep7['mes_ref'] = pd.to_datetime(df_rep7.DiaDeCompra).dt.to_period('M')
df_rep7 = df_rep7.loc[~df_rep7.Prefijo.isin(['OOH'])]


ValueError: Nenhum arquivo encontrado para o prefixo informado.

In [ ]:
df_prealoc = carregamento('4500 Preallocated Top Client_250326')

trocas = {
    '{GroupId}' : 'UserPS',
    '{IndividualId}' : 'individual-id',
    '{CreationTimeStamp}' : 'created_at'
}

df_prealoc.rename(columns=trocas, inplace=True)

df_prealoc['UserPS'] = df_prealoc['UserPS'].astype(str)

TypeError: carregamento() missing 1 required positional argument: 'path'

In [ ]:
merged = df_prealoc.merge(df_rep7, on='UserPS', how='left')

merged.UserPS = merged.UserPS.astype(int)
merged.UserPS = merged.UserPS - 550000000

In [4]:
from datetime import datetime
hoje = datetime.now().strftime("%d-%m-%Y")
hoje

'07-05-2026'

In [ ]:
merged.to_excel()

,UserPS,individual-id,created_at,Pais,Prefijo,Origen,IdIndividuo,DuenadeCasa,DiaDeCompra,NumCompras,Semana,OrigenGPM,ano_ref,len_us,mes_ref
0,2223489,0552223489-01,25/03/2026 15:24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
1,2223531,0552223531-01,25/03/2026 15:24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,2223651,0552223651-01,25/03/2026 15:24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3,2223477,0552223477-01,25/03/2026 15:24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4,2223299,0552223299-01,25/03/2026 15:24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4495,2230061,0552230061-01,25/03/2026 16:19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4496,2229908,0552229908-01,25/03/2026 16:19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4497,2229864,0552229864-01,25/03/2026 16:19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
4498,2230009,0552230009-01,25/03/2026 16:19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [3]:
def pegar_arquivo_mais_recente(pasta, extensao=None):
    import os

    """
    Retorna o caminho do arquivo mais recente dentro da pasta.
    
    :param pasta: caminho da pasta
    :param extensao: opcional (ex: '.csv', '.xlsx')
    :return: caminho completo do arquivo mais recente
    """
    
    arquivos = [
        os.path.join(pasta, f)
        for f in os.listdir(pasta)
        if os.path.isfile(os.path.join(pasta, f))
    ]
    
    # Filtrar por extensão (se informado)
    if extensao:
        arquivos = [f for f in arquivos if f.lower().endswith(extensao.lower())]
    
    if not arquivos:
        return None

    arquivo_mais_recente = max(arquivos, key=os.path.getctime)
    
    return arquivo_mais_recente

In [8]:
path = pegar_arquivo_mais_recente('c:/Users/70089581/Kantar/BKO - dados-ops/do_prealoc_atos', extensao=None)

In [ ]:
def disparar_email(path):

    import win32com.client as win32
    from datetime import datetime

    data_atual = datetime.now().strftime("%Y/%m/%d")

    outlook = win32.Dispatch('outlook.application')
    mail = outlook.CreateItem(0)

    mail.To = 'talita.cesar@wp.numerator.com; analucia.lopes@wp.numerator.com'
    mail.CC = 'luiz.farias@wp.numerator.com; fabio.shiraishi@wp.numerator.com'
    
    mail.Subject = f' TESTE 5 | {data_atual}'
    mail.Body = 'TESTE 5'

    #mail.Attachments.Add(path)

    mail.Send()

In [21]:
disparar_email(path)

# Ajuste visão ampla

In [5]:
from datetime import datetime

data_atual = datetime.now()
data_atual

datetime.datetime(2026, 5, 7, 11, 40, 34, 688761)

In [1]:
import functions 

pre_aloc = functions.carregamento(
        path='C:/Users/70089581/Kantar/BKO - dados-ops/do_preal_fornecedor/ref_prealoc',
        prefixo='pre')
    
trocas = {
'{GroupId}' : 'UserPS',
'{IndividualId}' : 'individual-id',
'{CreationTimeStamp}' : 'created_at'
}

pre_aloc.rename(columns=trocas, inplace=True)

pre_aloc['UserPS'] = pre_aloc['UserPS'].astype(str)

print("qty prealoca: ",pre_aloc.shape[0])

qty prealoca:  8500


In [2]:
rep7 = functions.carregamento(
        path='C:/Users/70089581/Kantar/BKO - dados-ops/do_rep7/bruto',
        prefixo='Brasil-Reporte-7'
    )

rep7 = functions.normalize(rep7)

functions.atualizar_base_criterio(pre_aloc,rep7)

Dimensão entrada saida iguais

atualizar_prealoc_atos: Arquivo Excel criado com sucesso! Salvo na pasta sincronizada do SharePoint 



# Analise Rápida

In [4]:
import functions
import pandas as pd

path = 'C:/Users/70089581/Kantar/BKO - dados-ops/do_rep7/'

df_1 = pd.read_excel(path + 'do_rep7_pstools_br_2026-01.xlsx')
df_2 = pd.read_excel(path + 'do_rep7_pstools_br_2026-02.xlsx')
df_3 = pd.read_excel(path + 'do_rep7_pstools_br_2026-03.xlsx')
df_4 = pd.read_excel(path + 'do_rep7_pstools_br_2026-04.xlsx')
df_5 = pd.read_excel(path + 'do_rep7_pstools_br_2026-05.xlsx')

df_rep7 = pd.concat([df_1,df_2,df_3,df_4,df_5])



In [8]:
df_rep7[df_rep7.UserPS == '552254335'].to_excel('output_552254335.xlsx')